In [5]:
# ─────────────────────────────────────────────
# Setup and imports
#
# PURPOSE: Load all libraries we need for exploration
# NOTE: This notebook is for DATA DISCOVERY only
#       We run this ONCE to understand the API
#       before designing any schema or pipeline code
# ─────────────────────────────────────────────

import sys
import json
import requests
import pandas as pd

# Base URL for FakeStoreAPI
# All endpoints are appended to this
API_BASE_URL = "https://fakestoreapi.com"

# Verify we are using the correct virtual environment
print(f"Python:   {sys.executable}")
print(f"requests: {requests.__version__}")
print(f"pandas:   {pd.__version__}")
print("Environment verified ✅")

Python:   c:\Users\phild\Desktop\Data Engineering\Retail_ETL_project\retail_etl\venv\Scripts\python.exe
requests: 2.31.0
pandas:   2.1.4
Environment verified ✅


In [6]:
# ─────────────────────────────────────────────
# Pull raw products from API
#
# PURPOSE: See exactly what the API sends back
#          before any cleaning or transformation
#
# WHY look at raw JSON first?
# The raw response shows us:
#   - what fields exist
#   - what types the values are
#   - whether anything is nested
#   - any surprises we didn't expect
#
# This is our source of truth for schema design
# ─────────────────────────────────────────────

response = requests.get(f"{API_BASE_URL}/products")
products_raw = response.json()

print(f"Status code:      {response.status_code}")
print(f"Records received: {len(products_raw)}")
print(f"\nFirst product — raw JSON:")
print(json.dumps(products_raw[0], indent=2))

Status code:      200
Records received: 20

First product — raw JSON:
{
  "id": 1,
  "title": "Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops",
  "price": 109.95,
  "description": "Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",
  "category": "men's clothing",
  "image": "https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png",
  "rating": {
    "rate": 3.9,
    "count": 120
  }
}


In [7]:
# ─────────────────────────────────────────────
# Convert products to DataFrame
#
# WHY pd.json_normalize() instead of pd.DataFrame()?
#
# pd.DataFrame(products_raw):
#   → keeps nested objects as dict/list inside a cell
#   → hard to profile, hard to query
#
# pd.json_normalize(products_raw):
#   → automatically flattens nested dicts
#   → e.g. {"rating": {"rate": 3.9, "count": 120}}
#      becomes two columns: rating.rate | rating.count
#   → gives us a true flat table to profile
# ─────────────────────────────────────────────

df_products = pd.json_normalize(products_raw)

print(f"Shape: {df_products.shape}  (rows, columns)")
print(f"\nColumns after normalisation:")
for col in df_products.columns:
    print(f"  {col}")

print(f"\nPreview:")
df_products.head()

Shape: (20, 8)  (rows, columns)

Columns after normalisation:
  id
  title
  price
  description
  category
  image
  rating.rate
  rating.count

Preview:


,id,title,price,description,category,image,rating.rate,rating.count
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,Your perfect pack for everyday use and walks i...,men's clothing,https://fakestoreapi.com/img/81fPKd-2AYL._AC_S...,3.9,120
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,"Slim-fitting style, contrast raglan long sleev...",men's clothing,https://fakestoreapi.com/img/71-3HjGNDUL._AC_S...,4.1,259
2,3,Mens Cotton Jacket,55.99,great outerwear jackets for Spring/Autumn/Wint...,men's clothing,https://fakestoreapi.com/img/71li-ujtlUL._AC_U...,4.7,500
3,4,Mens Casual Slim Fit,15.99,The color could be slightly different between ...,men's clothing,https://fakestoreapi.com/img/71YXzeOuslL._AC_U...,2.1,430
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,"From our Legends Collection, the Naga was insp...",jewelery,https://fakestoreapi.com/img/71pWzhdJNwL._AC_U...,4.6,400


In [8]:
# ─────────────────────────────────────────────
# Profile products DataFrame
#
# PURPOSE: Answer the key schema design questions:
#
# Data types  → what SQL type does each column need?
# Null counts → which columns need NOT NULL constraint?
# Unique vals → is this a key? a category? a free text?
#
# We document findings here so schema decisions
# are based on real data, not assumptions
# ─────────────────────────────────────────────

print("=" * 50)
print("DATA TYPES")
print("=" * 50)
print(df_products.dtypes)

print("\n" + "=" * 50)
print("NULL COUNTS (important for NOT NULL decisions)")
print("=" * 50)
nulls = df_products.isnull().sum()
for col, count in nulls.items():
    flag = " ← HAS NULLS — must be nullable in schema" if count > 0 else ""
    print(f"  {col:<30} {count} nulls{flag}")

print("\n" + "=" * 50)
print("UNIQUE VALUE COUNTS (important for key/category decisions)")
print("=" * 50)
for col in df_products.columns:
    unique = df_products[col].nunique()
    total = len(df_products)
    if unique == total:
        flag = " ← POTENTIAL PRIMARY KEY"
    elif unique <= 6:
        flag = " ← LIKELY CATEGORY/ENUM"
    else:
        flag = ""
    print(f"  {col:<30} {unique:>3} unique values{flag}")

print("\n" + "=" * 50)
print("SAMPLE VALUES (first 3 rows per column)")
print("=" * 50)
for col in df_products.columns:
    samples = df_products[col].head(3).tolist()
    print(f"  {col:<30} {samples}")

DATA TYPES
id                int64
title            object
price           float64
description      object
category         object
image            object
rating.rate     float64
rating.count      int64
dtype: object

NULL COUNTS (important for NOT NULL decisions)
  id                             0 nulls
  title                          0 nulls
  price                          0 nulls
  description                    0 nulls
  category                       0 nulls
  image                          0 nulls
  rating.rate                    0 nulls
  rating.count                   0 nulls

UNIQUE VALUE COUNTS (important for key/category decisions)
  id                              20 unique values ← POTENTIAL PRIMARY KEY
  title                           20 unique values ← POTENTIAL PRIMARY KEY
  price                           19 unique values
  description                     20 unique values ← POTENTIAL PRIMARY KEY
  category                         4 unique values ← LIKELY CATEGORY/EN

In [9]:
# ─────────────────────────────────────────────
# Pull raw carts from API
#
# PURPOSE: Understand the cart structure
#
# Carts are more complex than products because:
# - One cart belongs to ONE user
# - One cart contains MULTIPLE products
# - This is a one-to-many relationship
#
# Key question to answer here:
# How are products stored inside a cart?
# → as a nested list? → needs special handling
# ─────────────────────────────────────────────

response = requests.get(f"{API_BASE_URL}/carts")
carts_raw = response.json()

print(f"Status code:      {response.status_code}")
print(f"Records received: {len(carts_raw)}")
print(f"\nFirst cart — raw JSON:")
print(json.dumps(carts_raw[0], indent=2))

Status code:      200
Records received: 7

First cart — raw JSON:
{
  "id": 1,
  "userId": 1,
  "date": "2020-03-02T00:00:00.000Z",
  "products": [
    {
      "productId": 1,
      "quantity": 4
    },
    {
      "productId": 2,
      "quantity": 1
    },
    {
      "productId": 3,
      "quantity": 6
    }
  ],
  "__v": 0
}


In [10]:
# ─────────────────────────────────────────────
# Convert carts to DataFrame
#
# NOTE: json_normalize will flatten the top level
# but the 'products' column will still be a list
# because it contains multiple items per cart
#
# We will explore that nested list separately
# in the next cell
# ─────────────────────────────────────────────

df_carts = pd.json_normalize(carts_raw)

print(f"Shape: {df_carts.shape}  (rows, columns)")
print(f"\nColumns:")
for col in df_carts.columns:
    print(f"  {col}")

print(f"\nPreview:")
df_carts.head()

Shape: (7, 5)  (rows, columns)

Columns:
  id
  userId
  date
  products
  __v

Preview:


,id,userId,date,products,__v
0,1,1,2020-03-02T00:00:00.000Z,"[{'productId': 1, 'quantity': 4}, {'productId'...",0
1,2,1,2020-01-02T00:00:00.000Z,"[{'productId': 2, 'quantity': 4}, {'productId'...",0
2,3,2,2020-03-01T00:00:00.000Z,"[{'productId': 1, 'quantity': 2}, {'productId'...",0
3,4,3,2020-01-01T00:00:00.000Z,"[{'productId': 1, 'quantity': 4}]",0
4,5,3,2020-03-01T00:00:00.000Z,"[{'productId': 7, 'quantity': 1}, {'productId'...",0


In [11]:
# ─────────────────────────────────────────────
# Profile carts + explore nested list
#
# The 'products' column inside carts is special:
# It contains a LIST of dicts like:
# [
#   {"productId": 1, "quantity": 3},
#   {"productId": 4, "quantity": 2}
# ]
#
# This means ONE cart row has MULTIPLE products
# In our staging table we need to EXPLODE this:
# → one row per product per cart
# → this is called "flattening" or "normalising"
#
# Why explode it?
# Business question: "how many times was product X ordered?"
# → impossible to answer if products stay as a list ❌
# → easy to answer after exploding to one row per product ✅
# ─────────────────────────────────────────────

print("=" * 50)
print("DATA TYPES")
print("=" * 50)
print(df_carts.dtypes)

print("\n" + "=" * 50)
print("NULL COUNTS")
print("=" * 50)
for col, count in df_carts.isnull().sum().items():
    flag = " ← HAS NULLS" if count > 0 else ""
    print(f"  {col:<30} {count} nulls{flag}")

print("\n" + "=" * 50)
print("NESTED LIST — products inside cart")
print("=" * 50)
print("Raw products list from first cart:")
print(json.dumps(carts_raw[0]['products'], indent=2))

print("\nHow many products per cart?")
df_carts['product_count'] = df_carts['products'].apply(len)
print(df_carts['product_count'].describe())

DATA TYPES
id           int64
userId       int64
date        object
products    object
__v          int64
dtype: object

NULL COUNTS
  id                             0 nulls
  userId                         0 nulls
  date                           0 nulls
  products                       0 nulls
  __v                            0 nulls

NESTED LIST — products inside cart
Raw products list from first cart:
[
  {
    "productId": 1,
    "quantity": 4
  },
  {
    "productId": 2,
    "quantity": 1
  },
  {
    "productId": 3,
    "quantity": 6
  }
]

How many products per cart?
count    7.000000
mean     2.000000
std      0.816497
min      1.000000
25%      1.500000
50%      2.000000
75%      2.500000
max      3.000000
Name: product_count, dtype: float64


In [12]:
# ─────────────────────────────────────────────
# Pull raw users from API
#
# PURPOSE: Understand the user structure
#
# Users are complex because they have
# deeply nested objects:
# - name  → {firstname, lastname}
# - address → {street, city, zipcode, geolocation}
# - geolocation → {lat, long}  (nested inside address!)
#
# Key question: how deep does the nesting go?
# Each nested level = decisions about flattening
# ─────────────────────────────────────────────

response = requests.get(f"{API_BASE_URL}/users")
users_raw = response.json()

print(f"Status code:      {response.status_code}")
print(f"Records received: {len(users_raw)}")
print(f"\nFirst user — raw JSON:")
print(json.dumps(users_raw[0], indent=2))

Status code:      200
Records received: 10

First user — raw JSON:
{
  "address": {
    "geolocation": {
      "lat": "-37.3159",
      "long": "81.1496"
    },
    "city": "kilcoole",
    "street": "new road",
    "number": 7682,
    "zipcode": "12926-3874"
  },
  "id": 1,
  "email": "john@gmail.com",
  "username": "johnd",
  "password": "m38rmF$",
  "name": {
    "firstname": "john",
    "lastname": "doe"
  },
  "phone": "1-570-236-7033",
  "__v": 0
}


In [13]:
# ─────────────────────────────────────────────
# Convert users to DataFrame
#
# json_normalize will flatten ALL nested dicts:
# name.firstname, name.lastname
# address.street, address.city, address.zipcode
# address.geolocation.lat, address.geolocation.long
#
# This tells us exactly what columns
# our staging.users table needs
# ─────────────────────────────────────────────

df_users = pd.json_normalize(users_raw)

print(f"Shape: {df_users.shape}  (rows, columns)")
print(f"\nColumns after full normalisation:")
for col in df_users.columns:
    print(f"  {col}")

print(f"\nPreview:")
df_users.head()

Shape: (10, 14)  (rows, columns)

Columns after full normalisation:
  id
  email
  username
  password
  phone
  __v
  address.geolocation.lat
  address.geolocation.long
  address.city
  address.street
  address.number
  address.zipcode
  name.firstname
  name.lastname

Preview:


,id,email,username,password,phone,__v,address.geolocation.lat,address.geolocation.long,address.city,address.street,address.number,address.zipcode,name.firstname,name.lastname
0,1,john@gmail.com,johnd,m38rmF$,1-570-236-7033,0,-37.3159,81.1496,kilcoole,new road,7682,12926-3874,john,doe
1,2,morrison@gmail.com,mor_2314,83r5^_,1-570-236-7033,0,-37.3159,81.1496,kilcoole,Lovers Ln,7267,12926-3874,david,morrison
2,3,kevin@gmail.com,kevinryan,kev02937@,1-567-094-1345,0,40.3467,-30.1310,Cullman,Frances Ct,86,29567-1452,kevin,ryan
3,4,don@gmail.com,donero,ewedon,1-765-789-6734,0,50.3467,-20.1310,San Antonio,Hunters Creek Dr,6454,98234-1734,don,romer
4,5,derek@gmail.com,derek,jklg*_56,1-956-001-1945,0,40.3467,-40.1310,san Antonio,adams St,245,80796-1234,derek,powell


In [14]:
# ─────────────────────────────────────────────
# Profile users DataFrame
#
# Key things to look for:
# - email unique? → business identifier for a user
# - username unique? → another potential key
# - any nulls? → affects NOT NULL constraints
# - geolocation format? → NUMERIC precision needed
# ─────────────────────────────────────────────

print("=" * 50)
print("DATA TYPES")
print("=" * 50)
print(df_users.dtypes)

print("\n" + "=" * 50)
print("NULL COUNTS")
print("=" * 50)
for col, count in df_users.isnull().sum().items():
    flag = " ← HAS NULLS" if count > 0 else ""
    print(f"  {col:<30} {count} nulls{flag}")

print("\n" + "=" * 50)
print("UNIQUE VALUE COUNTS")
print("=" * 50)
for col in df_users.columns:
    unique = df_users[col].nunique()
    total = len(df_users)
    if unique == total:
        flag = " ← POTENTIAL KEY"
    elif unique <= 6:
        flag = " ← LIKELY CATEGORY"
    else:
        flag = ""
    print(f"  {col:<30} {unique:>3} unique{flag}")

print("\n" + "=" * 50)
print("SAMPLE VALUES (first 3 rows)")
print("=" * 50)
for col in df_users.columns:
    samples = df_users[col].head(3).tolist()
    print(f"  {col:<30} {samples}")

DATA TYPES
id                           int64
email                       object
username                    object
password                    object
phone                       object
__v                          int64
address.geolocation.lat     object
address.geolocation.long    object
address.city                object
address.street              object
address.number               int64
address.zipcode             object
name.firstname              object
name.lastname               object
dtype: object

NULL COUNTS
  id                             0 nulls
  email                          0 nulls
  username                       0 nulls
  password                       0 nulls
  phone                          0 nulls
  __v                            0 nulls
  address.geolocation.lat        0 nulls
  address.geolocation.long       0 nulls
  address.city                   0 nulls
  address.street                 0 nulls
  address.number                 0 nulls
  address.zipcode    